# IEEE 8500 — chunked dataset with **fixed controller initialization**

Same spacing / scenario rules as the past no-BESS diverse generator.

**Save location (same parent, different folder name):**

| | Path |
|--|------|
| Previous | .../datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40 |
| **This run** | .../datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40_fixedctrlinit |

Parent is auto-picked as the datasets_gnn2 that already contains the previous folder (H: / K: / Colab Drive).

| Knob | Value |
|------|-------|
| Scenarios x samples | 2000 x 40 |
| Chunk size | 50 scenarios |
| Time bins | load/pv/net = 3, include_anchors=True |
| Load/PV scale ranges | [0.3, 1.8] |
| sigma_load / sigma_pv | 0.5 |
| BESS | off |
| Node PE | k=8 from static edges |

**Only change vs the old workflow:** 
ixed_controller_init=True (reset TapNumber=0; CapControl banks OFF; fixed bank (CAPBank3) ON).


In [ ]:
# ============================================================
# CHUNKED RUN: scenarios x samples/scenario — NO BESS
# Same spacing as past diverse 2000×40, PLUS fixed controller init
# (reset: TapNumber=0; CapControl banks OFF; fixed bank ON).
# ============================================================
import os
import math
import time
from pathlib import Path

# ---------------- user controls ----------------
INCLUDE_BESS = False
FIXED_CONTROLLER_INIT = True  # ONLY behavioral change vs past diverse generator

TOTAL_SCENARIOS = 2000
N_SAMPLES_PER_SCENARIO = 40
SCENARIOS_PER_CHUNK = 50
BASE_SEED = 20320230

SIGMA_LOAD = 0.5
SIGMA_PV = 0.5

P_LOAD_MEAN_KW = 13731.9
Q_LOAD_MEAN_KVAR = 2610.15
P_LOAD_SCALE_RANGE = (0.3, 1.8)
Q_LOAD_SCALE_RANGE = (0.3, 1.8)
P_PV_SCALE_RANGE = (0.3, 1.8)

# Only used if INCLUDE_BESS is True
BESS_TOTAL_MVA_MEAN = 0
BESS_TOTAL_MVA_SIGMA = 0.1
BESS_NUM_NODES_MIN = 1
BESS_NUM_NODES_MAX = 3
BESS_Q_FRAC_MAX = 0.44
BESS_CANDIDATE_NODES_150 = []  # fill if you set INCLUDE_BESS = True

VMIN_SAFE_PU = 0.55
VMAX_SAFE_PU = 1.45

NODE_PE_K = 8
NODE_PE_SEED = 42
NODE_PE_ZERO_EIG_TOL = 1e-8

RETURN_NODE_DF = False

# Same parent directory as the previous diverse dataset; different folder name only.
# Previous:  .../datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40
# This run:  .../datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40_fixedctrlinit
_PREV_NAME = "original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40"
_CHUNK_NAME = _PREV_NAME + "_fixedctrlinit"
_PARENT_CANDIDATES = [
    Path(r"H:\My Drive\datasets_gnn2"),
    Path(r"K:\My Drive\datasets_gnn2"),
    Path("/content/drive/MyDrive/datasets_gnn2"),
    Path(r"C:\Users\alita\OneDrive\Desktop\GNN2\datasets_gnn2"),
]
# Prefer the parent that already holds the previous dataset folder.
DATASETS_GNN2 = next(
    (p for p in _PARENT_CANDIDATES if (p / _PREV_NAME).is_dir()),
    next((p for p in _PARENT_CANDIDATES if p.is_dir()), _PARENT_CANDIDATES[0]),
)
CHUNK_ROOT = DATASETS_GNN2 / _CHUNK_NAME
CHUNK_ROOT.mkdir(parents=True, exist_ok=True)
print("PREVIOUS dataset:", DATASETS_GNN2 / _PREV_NAME)
print("THIS dataset:    ", CHUNK_ROOT)

CANDIDATES = [
    r"C:\Users\alita\OneDrive\Desktop\GNN2",
    "/content/GNN-Sandia",
    "/content/GNN2",
    os.getcwd(),
]

script_path = None
for root in CANDIDATES:
    p = os.path.join(root, "run_original_style_dataset_8500_unbalanced.py")
    if os.path.isfile(p):
        script_path = os.path.abspath(p)
        break

if script_path is None:
    raise FileNotFoundError("run_original_style_dataset_8500_unbalanced.py not found in candidates.")

os.chdir(os.path.dirname(script_path))
print("CWD:", os.getcwd())
print("SCRIPT:", script_path)
print("CHUNK_ROOT:", CHUNK_ROOT)
print("FIXED_CONTROLLER_INIT:", FIXED_CONTROLLER_INIT)

ns = {"__name__": "run_original_style_dataset_8500_unbalanced", "__file__": script_path}
exec(open(script_path, encoding="utf-8").read(), ns)

root = CHUNK_ROOT
if not root.exists():
    raise FileNotFoundError(f"CHUNK_ROOT not found: {root}")

n_chunks = math.ceil(TOTAL_SCENARIOS / SCENARIOS_PER_CHUNK)
print(f"\nTotal chunks: {n_chunks} | INCLUDE_BESS={INCLUDE_BESS} | FIXED_CONTROLLER_INIT={FIXED_CONTROLLER_INIT}")

for chunk_idx in range(n_chunks):
    s0 = chunk_idx * SCENARIOS_PER_CHUNK
    n_this = min(SCENARIOS_PER_CHUNK, TOTAL_SCENARIOS - s0)
    chunk_seed = int(BASE_SEED + 100003 * (chunk_idx + 1))
    out_dir = CHUNK_ROOT / f"run_{chunk_idx+1:03d}_scen_{s0:04d}_{s0+n_this-1:04d}_seed_{chunk_seed}"
    out_dir.mkdir(parents=True, exist_ok=True)

    ns["OUT_DIR"] = out_dir
    ns["EDGE_CSV"] = out_dir / "gnn_edges_phase_static.csv"
    ns["NODE_CSV"] = out_dir / "gnn_node_features_and_targets.csv"
    ns["SAMPLE_CSV"] = out_dir / "gnn_sample_meta.csv"
    ns["NODE_INDEX_CSV"] = out_dir / "gnn_node_index_master.csv"

    print(f"\n=== Chunk {chunk_idx+1}/{n_chunks} ===")
    print(f"Scenarios in chunk: {n_this}")
    print(f"Seed: {chunk_seed}")
    print(f"OUT_DIR: {out_dir}")

    gen_kw = dict(
        n_scenarios=int(n_this),
        k_snapshots_per_scenario_total=int(N_SAMPLES_PER_SCENARIO),
        bins_by_profile={"load": 3, "pv": 3, "net": 3},
        include_anchors=True,
        master_seed=int(chunk_seed),
        sigma_load=float(SIGMA_LOAD),
        sigma_pv=float(SIGMA_PV),
        p_load_mean_kw=float(P_LOAD_MEAN_KW),
        q_load_mean_kvar=float(Q_LOAD_MEAN_KVAR),
        p_load_scale_range=tuple(P_LOAD_SCALE_RANGE),
        q_load_scale_range=tuple(Q_LOAD_SCALE_RANGE),
        p_pv_scale_range=tuple(P_PV_SCALE_RANGE),
        vmin_safe_pu=float(VMIN_SAFE_PU),
        vmax_safe_pu=float(VMAX_SAFE_PU),
        include_source_in_safe_band=True,
        return_node_df=bool(RETURN_NODE_DF),
        node_pe_k=int(NODE_PE_K),
        node_pe_seed=int(NODE_PE_SEED),
        node_pe_zero_eig_tol=float(NODE_PE_ZERO_EIG_TOL),
        include_bess=bool(INCLUDE_BESS),
        fixed_controller_init=bool(FIXED_CONTROLLER_INIT),
    )

    if INCLUDE_BESS:
        if not BESS_CANDIDATE_NODES_150:
            raise ValueError("INCLUDE_BESS=True requires a non-empty BESS_CANDIDATE_NODES_150 list.")
        gen_kw.update(
            bess_total_mva_mean=float(BESS_TOTAL_MVA_MEAN),
            bess_total_mva_sigma=float(BESS_TOTAL_MVA_SIGMA),
            bess_num_nodes_min=int(BESS_NUM_NODES_MIN),
            bess_num_nodes_max=int(BESS_NUM_NODES_MAX),
            bess_q_frac_max=float(BESS_Q_FRAC_MAX),
            bess_candidate_nodes_override=BESS_CANDIDATE_NODES_150,
        )

    t0 = time.time()
    df_sample, df_node = ns["generate_original_style_dataset_8500_unbalanced"](**gen_kw)

    dt = time.time() - t0
    n_kept = int(df_sample["sample_id"].nunique()) if len(df_sample) else 0
    print(f"Chunk done in {dt/60:.1f} min | kept samples: {n_kept} | rows sample_meta: {len(df_sample)}")
    if len(df_sample) and "fixed_controller_init" in df_sample.columns:
        print(
            "fixed_controller_init column:",
            sorted(df_sample["fixed_controller_init"].astype(int).unique().tolist()),
        )
    print("Saved:")
    print(" -", ns["NODE_INDEX_CSV"])
    print(" -", ns["EDGE_CSV"])
    print(" -", ns["SAMPLE_CSV"])
    print(" -", ns["NODE_CSV"])

print("\nAll chunks finished.")


### Stamp draft Y-edges (sibling `*_yedges`)
Copies the fixedctrlinit chunk parent and replaces only `gnn_edges_phase_static.csv` with network-Y edges (`R_full/X_full = Re(Y)/Im(Y)`). Source dataset is **not** modified (`inplace=False`).

Needs OpenDSS + the feeder folder `8500 nodes with solar unbalanced/` (Master-PV2MW-inv.dss).

On **Colab**, sparse checkouts often omit that folder and `git checkout …` fails with `pathspec did not match`. The stamp cell downloads the feeder from GitHub if git restore fails. Or run this paste cell once:

```python
import json, urllib.request
from pathlib import Path
REPO = Path('/content/GNN-Sandia')
DST = REPO / '8500 nodes with solar unbalanced'
DST.mkdir(parents=True, exist_ok=True)
api = 'https://api.github.com/repos/alitasavori/GNN-Sandia/contents/8500%20nodes%20with%20solar%20unbalanced'
for item in json.load(urllib.request.urlopen(urllib.request.Request(api, headers={'User-Agent':'x'}))):
    if item.get('type')!='file' or not item.get('download_url'): continue
    p = DST / item['name']
    print(item['name']); p.write_bytes(urllib.request.urlopen(item['download_url']).read())
print('OK', (DST/'Master-PV2MW-inv.dss').is_file())
```

Prefer stamping locally on H:/ or K: when possible (Drive copy of 40 chunks is slow).


In [ ]:
# ============================================================
# 8500 Y-edge sibling from FIXEDCTRLINIT chunks (stamp_network_y_edges)
# Source:  ..._2000_40_fixedctrlinit
# Output:  ..._2000_40_fixedctrlinit_yedges
# ============================================================
from __future__ import annotations

import importlib
import shutil
import subprocess
import sys
from pathlib import Path


def _find_repo(mod: str = "stamp_network_y_edges.py") -> Path:
    cands: list[Path] = []
    try:
        cands.append(Path(__file__).resolve().parent)
    except NameError:
        pass
    cwd = Path.cwd().resolve()
    cands.append(cwd)
    cands.extend(cwd.parents)
    cands.extend(
        [
            Path(r"C:\Users\alita\OneDrive\Desktop\GNN2"),
            Path("/content/GNN-Sandia"),
            Path("/content/GNN2"),
        ]
    )
    seen: set[Path] = set()
    for root in cands:
        root = Path(root).resolve()
        if root in seen or not root.exists():
            continue
        seen.add(root)
        if (root / mod).is_file():
            return root
    raise FileNotFoundError(f"Could not find {mod}. Open notebook from GNN2 / GNN-Sandia repo.")


def _ensure_8500_dss(repo: Path) -> Path:
    """Colab sparse clones often omit the feeder DSS folder; restore it."""
    import json
    import urllib.request

    model = repo / "8500 nodes with solar unbalanced"
    master = model / "Master-PV2MW-inv.dss"
    if master.is_file():
        return master

    print(f"[stamp-y] missing {master}")

    # 1) Disable sparse-checkout if present, then checkout from origin/main
    try:
        sc = repo / ".git" / "info" / "sparse-checkout"
        if sc.is_file() or (repo / ".git" / "config").is_file():
            subprocess.run(
                ["git", "sparse-checkout", "disable"],
                cwd=str(repo),
                check=False,
                capture_output=True,
                text=True,
            )
        subprocess.run(["git", "fetch", "origin", "main"], cwd=str(repo), check=False)
        for ref in ("origin/main", "FETCH_HEAD", "HEAD"):
            r = subprocess.run(
                ["git", "checkout", ref, "--", "8500 nodes with solar unbalanced"],
                cwd=str(repo),
                check=False,
                capture_output=True,
                text=True,
            )
            print(f"[stamp-y] git checkout {ref} rc={r.returncode}")
            if master.is_file():
                print("[stamp-y] restored DSS via git")
                return master
    except Exception as exc:  # noqa: BLE001
        print("[stamp-y] git restore failed:", exc)

    # 2) Download folder from GitHub API (bypasses sparse-checkout)
    api = (
        "https://api.github.com/repos/alitasavori/GNN-Sandia/contents/"
        "8500%20nodes%20with%20solar%20unbalanced"
    )
    print(f"[stamp-y] downloading feeder from GitHub API …")
    try:
        req = urllib.request.Request(api, headers={"User-Agent": "GNN-Sandia-stamp"})
        with urllib.request.urlopen(req, timeout=120) as resp:
            listing = json.loads(resp.read().decode("utf-8"))
        if not isinstance(listing, list):
            raise RuntimeError(f"unexpected API payload: {type(listing)}")
        model.mkdir(parents=True, exist_ok=True)
        for item in listing:
            if item.get("type") != "file":
                continue
            name = item["name"]
            url = item.get("download_url")
            if not url:
                continue
            dest = model / name
            print(f"  -> {name}")
            with urllib.request.urlopen(url, timeout=120) as r2, open(dest, "wb") as f:
                f.write(r2.read())
    except Exception as exc:  # noqa: BLE001
        print("[stamp-y] GitHub download failed:", exc)

    if master.is_file():
        print("[stamp-y] restored DSS via GitHub download")
        return master

    # 3) Optional: copy from Drive mirrors
    drive_cands = [
        Path("/content/drive/MyDrive/datasets_gnn2/8500 nodes with solar unbalanced"),
        Path("/content/drive/MyDrive/GNN2/8500 nodes with solar unbalanced"),
        Path("/content/drive/MyDrive/GNN-Sandia/8500 nodes with solar unbalanced"),
    ]
    for src_dir in drive_cands:
        if (src_dir / "Master-PV2MW-inv.dss").is_file():
            print(f"[stamp-y] copying DSS from Drive: {src_dir}")
            if model.exists():
                shutil.rmtree(model, ignore_errors=True)
            shutil.copytree(src_dir, model)
            break

    if not master.is_file():
        raise FileNotFoundError(
            f"Missing OpenDSS master:\n  {master}\n"
            "On Colab run this cell (downloads feeder from GitHub), or:\n"
            "  !mkdir -p '/content/GNN-Sandia/8500 nodes with solar unbalanced'\n"
            "  then re-run stamp; or stamp locally on H:/K:.\n"
        )
    return master


_REPO = _find_repo()
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))
print("REPO:", _REPO)
_ensure_8500_dss(_REPO)

import stamp_network_y_edges as stamp

stamp = importlib.reload(stamp)

SRC_NAME = "original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40_fixedctrlinit"
OUT_NAME = SRC_NAME + "_yedges"

_PARENT_CANDIDATES = [
    Path(r"H:\My Drive\datasets_gnn2"),
    Path(r"K:\My Drive\datasets_gnn2"),
    Path("/content/drive/MyDrive/datasets_gnn2"),
    _REPO / "datasets_gnn2",
]

try:
    import google.colab  # noqa: F401

    ON_COLAB = True
except ImportError:
    ON_COLAB = False

DATA = next((p for p in _PARENT_CANDIDATES if (p / SRC_NAME).is_dir()), None)
if DATA is None:
    DATA = next((p for p in _PARENT_CANDIDATES if p.is_dir()), _PARENT_CANDIDATES[0])

SRC = DATA / SRC_NAME
OUT = DATA / OUT_NAME

print("ON_COLAB:", ON_COLAB)
print("SRC:", SRC)
print("OUT:", OUT)

if not SRC.is_dir():
    raise FileNotFoundError(
        f"Source chunk parent missing:\n  {SRC}\n"
        "Finish the FIXEDCTRLINIT chunked generation cell first, then re-run this stamp."
    )

if ON_COLAB:
    print(
        "NOTE: Stamping on Colab copies whole chunks through Drive (slow). "
        "Local H:/K: is usually faster; either works once DSS is present."
    )

info = stamp.stamp_chunk_parent(
    feeder="8500",
    chunk_parent=SRC,
    out_chunk_parent=OUT,
    inplace=False,
    dry_run=False,
)
print("Done:", info)
print("Next: run the MT-GPS train cell below with CHUNK_PARENT =", OUT)


### Build `*_mvagg.csv` (required before MT-GPS train)

CCE training reads `gnn_node_features_and_targets_mvagg.csv` (SX loads rolled up onto MV nodes).  
The fixedctrlinit generator only wrote the raw `gnn_node_features_and_targets.csv`.

Run this once on the stamped sibling (or the source). It is the same SX→MV rollup used for the original `…_2000_40` dataset.


In [ ]:
# ============================================================
# Build gnn_node_features_and_targets_mvagg.csv for each chunk
# (SX load → MV rollup; same as original 8500 CCE training data)
# ============================================================
from __future__ import annotations

import os
import re
from pathlib import Path

import pandas as pd

CHUNK_NAME = "original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40_fixedctrlinit_yedges"
# Also fix the ohm-edge source if you still need it:
ALSO_PARENTS = [
    "original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40_fixedctrlinit",
]

DELETE_RAW_AFTER = False  # keep raw; yedges stamp already copied it
SKIP_IF_EXISTS = True


def _find_repo() -> Path:
    cands = [
        Path(os.environ.get("GNN2_REPO_ROOT", "")),
        Path.cwd(),
        Path("/content/GNN-Sandia"),
        Path("/content/GNN2"),
        Path(r"C:\Users\alita\OneDrive\Desktop\GNN2"),
    ]
    for root in cands:
        if not root or not str(root):
            continue
        root = root.resolve()
        if (root / "8500 nodes with solar unbalanced" / "LoadXfmrs.dss").is_file():
            return root
        if (root / "train_da_gps_multitask_complex_voltage_gine.py").is_file():
            # may still miss DSS on Colab sparse clone
            dss = root / "8500 nodes with solar unbalanced" / "LoadXfmrs.dss"
            if dss.is_file():
                return root
    raise FileNotFoundError("Repo with LoadXfmrs.dss not found")


def _data_parents() -> list[Path]:
    bases = [
        Path("/content/drive/MyDrive/datasets_gnn2"),
        Path(r"H:\My Drive\datasets_gnn2"),
        Path(r"K:\My Drive\datasets_gnn2"),
    ]
    return [b for b in bases if b.is_dir()]


def _tok(s: str) -> str:
    return str(s).strip().lower()


def _bus_base(node_or_bus: str) -> str:
    return _tok(node_or_bus).split(".")[0]


def build_tx_map(tx_dss: Path) -> pd.DataFrame:
    wdg_bus_re = re.compile(r"wdg=(\d+)\s+bus=([^\s]+)", re.I)
    map_rows: list[tuple[str, str]] = []
    for line in tx_dss.read_text(encoding="utf-8", errors="ignore").splitlines():
        s = line.strip()
        if not s.lower().startswith("new transformer."):
            continue
        w = {int(k): _tok(v) for k, v in wdg_bus_re.findall(s)}
        if not (1 in w and 2 in w and 3 in w):
            continue
        x2 = _bus_base(w[2])
        x3 = _bus_base(w[3])
        if not (x2.startswith("x") and x3.startswith("x")):
            continue
        mv_node = w[1]
        for sx_bus in {_tok("s" + x2), _tok("s" + x3)}:
            map_rows.append((mv_node, sx_bus))
    map_long = pd.DataFrame(map_rows, columns=["mv_node", "sx_bus"]).drop_duplicates()
    print("TX map: unique MV=", map_long["mv_node"].nunique(), "rows=", len(map_long))
    return map_long


def build_mvagg_for_chunk(chunk_dir: Path, map_long: pd.DataFrame) -> dict:
    in_csv = chunk_dir / "gnn_node_features_and_targets.csv"
    idx_csv = chunk_dir / "gnn_node_index_master.csv"
    out_compat = chunk_dir / "gnn_node_features_and_targets_mvagg.csv"

    if SKIP_IF_EXISTS and out_compat.is_file():
        return {"skipped": True, "path": str(out_compat)}

    if not in_csv.is_file():
        raise FileNotFoundError(f"Missing raw node CSV: {in_csv}")
    if not idx_csv.is_file():
        raise FileNotFoundError(f"Missing node index: {idx_csv}")

    df = pd.read_csv(in_csv)
    idx = pd.read_csv(idx_csv, usecols=["node"])
    allowed_nodes = set(idx["node"].astype(str).str.strip().str.lower())

    df["node_lc"] = df["node"].astype(str).str.strip().str.lower()
    df["bus"] = df["node_lc"].str.split(".").str[0]

    sx = df[df["bus"].str.startswith("sx")][["sample_id", "bus", "p_load_kw", "q_load_kvar"]].copy()
    sx = sx.rename(columns={"bus": "sx_bus"})
    sx_join = sx.merge(map_long, on="sx_bus", how="inner")
    mv_agg = (
        sx_join.groupby(["sample_id", "mv_node"], as_index=False)[["p_load_kw", "q_load_kvar"]]
        .sum()
        .rename(columns={"p_load_kw": "p_load_kw_agg", "q_load_kvar": "q_load_kvar_agg"})
    )

    df2 = df[~(df["bus"].str.startswith("x") | df["bus"].str.startswith("sx"))].copy()
    df2 = df2[df2["node_lc"].isin(allowed_nodes)].copy()
    df2["p_load_kw"] = 0.0
    df2["q_load_kvar"] = 0.0
    df2 = df2.merge(
        mv_agg,
        left_on=["sample_id", "node_lc"],
        right_on=["sample_id", "mv_node"],
        how="left",
    )
    hit = df2["p_load_kw_agg"].notna()
    df2.loc[hit, "p_load_kw"] = df2.loc[hit, "p_load_kw_agg"]
    df2.loc[hit, "q_load_kvar"] = df2.loc[hit, "q_load_kvar_agg"]
    for c in ["node_lc", "bus", "mv_node", "p_load_kw_agg", "q_load_kvar_agg"]:
        if c in df2.columns:
            df2.drop(columns=c, inplace=True)

    # No-BESS: fold zero BESS cols away if present (compat with withder schema)
    compat = df2.copy()
    if "p_bess_kw" in compat.columns:
        compat["p_load_kw"] = compat["p_load_kw"].astype(float) + compat["p_bess_kw"].fillna(0.0).astype(float)
        compat["q_load_kvar"] = compat["q_load_kvar"].astype(float) + compat["q_bess_kvar"].fillna(0.0).astype(float)
        compat.drop(columns=["p_bess_kw", "q_bess_kvar"], inplace=True, errors="ignore")

    compat.to_csv(out_compat, index=False)
    if DELETE_RAW_AFTER and in_csv.is_file():
        in_csv.unlink()

    n_samples = int(compat["sample_id"].nunique()) if len(compat) else 0
    return {"skipped": False, "rows": len(compat), "samples": n_samples, "path": str(out_compat)}


REPO = _find_repo()
TX_DSS = REPO / "8500 nodes with solar unbalanced" / "LoadXfmrs.dss"
if not TX_DSS.is_file():
    # Colab sparse: try 8500-node mapping CSV path + download LoadXfmrs if needed
    raise FileNotFoundError(
        f"Missing {TX_DSS}\n"
        "On Colab, restore feeder DSS first (same as stamp), then re-run."
    )

map_long = build_tx_map(TX_DSS)
parents_data = _data_parents()
names = [CHUNK_NAME] + [n for n in ALSO_PARENTS if n != CHUNK_NAME]

for name in names:
    parent = next((b / name for b in parents_data if (b / name).is_dir()), None)
    if parent is None:
        print(f"[skip] not found: {name}")
        continue
    runs = sorted(p for p in parent.iterdir() if p.is_dir() and p.name.startswith("run_"))
    print(f"\n=== {parent}  ({len(runs)} chunks) ===")
    for rd in runs:
        info = build_mvagg_for_chunk(rd, map_long)
        if info.get("skipped"):
            print(f"  skip exists: {rd.name}")
        else:
            print(f"  wrote {rd.name}: rows={info['rows']} samples={info['samples']}")

print("\nDone. Re-run the MT-GPS train cell.")


### Train MT-GPS (CCE) on fixedctrlinit + draft Y-edges

Same architecture / losses as the shipped CCE run
(`da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE`:
h=96, L=2, heads=2, node_emb=4, `global_attn_mode=tokens`, `reg_loss=ce`, λ_cap/reg/pv=0.1).

**Data:** `original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40_fixedctrlinit_yedges`
(fixed controller init labels + draft network-Y edges).

Caches / run dirs use a **`fixedctrlinit_yedges`** suffix so they do not collide with the older warm-start `*_yedges` caches.

Prerequisite: finish the stamp cell above (sibling folder must exist on Drive / H: / K:).


In [ ]:
# ============================================================
# MT-GPS (CCE) train on FIXEDCTRLINIT + draft Y-edges
# Data: ..._2000_40_fixedctrlinit_yedges
# Arch: same as da_gps_chunked_l4_mvagg_gine_metaaux_regce_..._CCE
# ============================================================
from __future__ import annotations

import datetime
import os
import subprocess
import sys
import warnings
from pathlib import Path

CHUNK_NAME = "original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40_fixedctrlinit_yedges"

DRIVE_ROOT = Path("/content/drive")
MYDRIVE_DATA = DRIVE_ROOT / "MyDrive/datasets_gnn2"
COLAB_CHUNK_DEFAULT = MYDRIVE_DATA / CHUNK_NAME
WIN_CHUNK_CANDIDATES = [
    Path(r"H:\My Drive\datasets_gnn2") / CHUNK_NAME,
    Path(r"K:\My Drive\datasets_gnn2") / CHUNK_NAME,
    Path(r"D:\datasets") / CHUNK_NAME,
]


def _is_windows_drive_path(path) -> bool:
    s = str(path).strip().replace("/", "\\")
    return len(s) >= 2 and s[1] == ":" and s[0].isalpha()


def _on_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _drive_mounted() -> bool:
    return DRIVE_ROOT.is_dir() and (DRIVE_ROOT / "MyDrive").is_dir()


def _find_repo() -> Path:
    mod = "train_da_gps_multitask_complex_voltage_gine.py"
    cands: list[Path] = []
    env = os.environ.get("GNN2_REPO_ROOT", "").strip()
    if env:
        cands.append(Path(env))
    cwd = Path.cwd().resolve()
    cands.append(cwd)
    cands.extend(cwd.parents)
    cands.extend(
        [
            Path(r"C:\Users\alita\OneDrive\Desktop\GNN2"),
            Path("/content/GNN-Sandia"),
            Path("/content/GNN2"),
        ]
    )
    seen: set[Path] = set()
    for root in cands:
        root = Path(root).expanduser().resolve()
        if root in seen or not root.exists():
            continue
        seen.add(root)
        if (root / mod).is_file():
            return root
    raise FileNotFoundError(f"Could not find {mod}. Set GNN2_REPO_ROOT or cd into the repo.")


def _resolve_data_path(path: Path, *, label: str, colab_fallback: Path | None = None) -> Path:
    raw = str(path)
    if _is_windows_drive_path(raw):
        if _on_colab() and colab_fallback is not None:
            warnings.warn(
                f"{label}={raw!r} is a Windows path on Linux/Colab; using {colab_fallback} instead.",
                UserWarning,
                stacklevel=2,
            )
            return colab_fallback.expanduser().resolve()
        if _on_colab():
            raise ValueError(
                f"{label}={raw!r} is a Windows absolute path and invalid on Colab. "
                f"Mount Drive and use e.g. {COLAB_CHUNK_DEFAULT}"
            )
        return Path(raw).expanduser().resolve()
    p = Path(path).expanduser()
    if p.is_absolute():
        return p.resolve()
    return (Path.cwd() / p).resolve()


def _find_node_pe_csv(chunk_parent: Path) -> Path:
    hits = sorted(chunk_parent.glob("run_*/gnn_node_index_master.csv"))
    if not hits:
        raise FileNotFoundError(f"No run_*/gnn_node_index_master.csv under {chunk_parent}")
    return hits[0]


def _smoke_chunk_subdir_glob(chunk_parent: Path, smoke_count: int) -> str:
    runs = sorted(
        (p for p in chunk_parent.iterdir() if p.is_dir() and p.name.startswith("run_")),
        key=lambda p: p.name,
    )
    if len(runs) < smoke_count:
        raise ValueError(f"Need >= {smoke_count} run_* under {chunk_parent}, found {len(runs)}")
    kept = runs[:smoke_count]
    print(f"SMOKE: keeping first {smoke_count} chunks: {[p.name for p in kept]}")
    return ",".join(p.name for p in kept)



def _ensure_mvagg_under(chunk_parent: Path, repo: Path) -> None:
    """Old ..._2000_40 already had mvagg; fixedctrlinit gen omitted it. Build if missing."""
    runs = sorted(p for p in chunk_parent.iterdir() if p.is_dir() and p.name.startswith("run_"))
    missing = [
        rd for rd in runs
        if not (rd / "gnn_node_features_and_targets_mvagg.csv").is_file()
    ]
    if not missing:
        return
    print(f"[mvagg] {len(missing)}/{len(runs)} chunks missing mvagg; building …")
    import re
    import urllib.request
    import json as _json
    import pandas as pd

    tx = repo / "8500 nodes with solar unbalanced" / "LoadXfmrs.dss"
    if not tx.is_file():
        tx.parent.mkdir(parents=True, exist_ok=True)
        api = (
            "https://api.github.com/repos/alitasavori/GNN-Sandia/contents/"
            "8500%20nodes%20with%20solar%20unbalanced"
        )
        req = urllib.request.Request(api, headers={"User-Agent": "GNN-Sandia"})
        for item in _json.load(urllib.request.urlopen(req)):
            if item.get("type") != "file" or not item.get("download_url"):
                continue
            dest = tx.parent / item["name"]
            if dest.is_file():
                continue
            print("[mvagg] dl", item["name"])
            dest.write_bytes(urllib.request.urlopen(item["download_url"]).read())
    if not tx.is_file():
        raise FileNotFoundError(f"Need {tx} to build mvagg")

    def _tok(s: str) -> str:
        return str(s).strip().lower()

    def _bus_base(s: str) -> str:
        return _tok(s).split(".")[0]

    wdg_re = re.compile(r"wdg=(\d+)\s+bus=([^\s]+)", re.I)
    rows: list[tuple[str, str]] = []
    for line in tx.read_text(encoding="utf-8", errors="ignore").splitlines():
        s = line.strip()
        if not s.lower().startswith("new transformer."):
            continue
        w = {int(k): _tok(v) for k, v in wdg_re.findall(s)}
        if not (1 in w and 2 in w and 3 in w):
            continue
        x2, x3 = _bus_base(w[2]), _bus_base(w[3])
        if not (x2.startswith("x") and x3.startswith("x")):
            continue
        for sx in {_tok("s" + x2), _tok("s" + x3)}:
            rows.append((w[1], sx))
    map_long = pd.DataFrame(rows, columns=["mv_node", "sx_bus"]).drop_duplicates()
    print(f"[mvagg] building under {chunk_parent} (map={len(map_long)} rows)")

    # runs already computed above for missing check
    for i, rd in enumerate(runs, 1):
        out = rd / "gnn_node_features_and_targets_mvagg.csv"
        raw = rd / "gnn_node_features_and_targets.csv"
        idx = rd / "gnn_node_index_master.csv"
        if out.is_file():
            continue
        if not raw.is_file() or not idx.is_file():
            raise FileNotFoundError(f"Missing raw/index for mvagg in {rd}")
        df = pd.read_csv(raw)
        allowed = set(pd.read_csv(idx, usecols=["node"])["node"].astype(str).str.strip().str.lower())
        df["node_lc"] = df["node"].astype(str).str.strip().str.lower()
        df["bus"] = df["node_lc"].str.split(".").str[0]
        sx = df[df["bus"].str.startswith("sx")][["sample_id", "bus", "p_load_kw", "q_load_kvar"]].rename(
            columns={"bus": "sx_bus"}
        )
        mv_agg = (
            sx.merge(map_long, on="sx_bus", how="inner")
            .groupby(["sample_id", "mv_node"], as_index=False)[["p_load_kw", "q_load_kvar"]]
            .sum()
            .rename(columns={"p_load_kw": "p_a", "q_load_kvar": "q_a"})
        )
        df2 = df[~(df["bus"].str.startswith("x") | df["bus"].str.startswith("sx"))].copy()
        df2 = df2[df2["node_lc"].isin(allowed)].copy()
        df2["p_load_kw"] = 0.0
        df2["q_load_kvar"] = 0.0
        df2 = df2.merge(
            mv_agg, left_on=["sample_id", "node_lc"], right_on=["sample_id", "mv_node"], how="left"
        )
        hit = df2["p_a"].notna()
        df2.loc[hit, "p_load_kw"] = df2.loc[hit, "p_a"]
        df2.loc[hit, "q_load_kvar"] = df2.loc[hit, "q_a"]
        df2.drop(
            columns=[c for c in ["node_lc", "bus", "mv_node", "p_a", "q_a"] if c in df2.columns],
            inplace=True,
        )
        if "p_bess_kw" in df2.columns:
            df2["p_load_kw"] = df2["p_load_kw"].astype(float) + df2["p_bess_kw"].fillna(0.0).astype(float)
            df2["q_load_kvar"] = df2["q_load_kvar"].astype(float) + df2["q_bess_kvar"].fillna(0.0).astype(
                float
            )
            df2.drop(columns=["p_bess_kw", "q_bess_kvar"], inplace=True, errors="ignore")
        df2.to_csv(out, index=False)
        print(f"[mvagg] [{i}/{len(runs)}] {rd.name} rows={len(df2)}")
    print("[mvagg] done")



if _on_colab() and not _drive_mounted():
    from google.colab import drive

    drive.mount("/content/drive")

# --- smoke / full-run toggles ---
SMOKE_TEST = False
SMOKE_CHUNK_COUNT = 3
SMOKE_EPOCHS = 15
SMOKE_PATIENCE = 5
SMOKE_SEED = 42

# Separate caches from warm-start yedges / ohm CCE (labels differ under fixedctrlinit)
_DA_CACHE_NAME = (
    "da_gps_chunked_mvagg_fixedctrlinit_yedges_smoke_gine"
    if SMOKE_TEST
    else "da_gps_chunked_mvagg_fixedctrlinit_yedges_full_gine"
)
_GNN_CACHE_NAME = "gnn_only_chunked_mvagg_fixedctrlinit_yedges_full_gine"

# --- CCE architecture (from training_last.pt) ---
CCE_HIDDEN = 96
CCE_LAYERS = 2
CCE_HEADS = 2
CCE_NODE_EMB_DIM = 4
META_AUX_COLS = "pv_pv2_p_post_kw,pv_pv2_q_post_kvar,P_loss_total_post_kw,Q_loss_total_post_kvar"
NUM_WORKERS = 0 if os.name == "nt" else 4
BATCH_SIZE = 64

LAMBDA_CAP = "0.1"
LAMBDA_REG = "0.1"
LAMBDA_PV = "0.1"
GLOBAL_ATTN_MODE = "tokens"

REPO = _find_repo()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["GNN2_REPO_ROOT"] = str(REPO)

if _on_colab():
    if not _drive_mounted():
        raise RuntimeError("Colab requires Google Drive mounted.")
    CHUNK_PARENT = COLAB_CHUNK_DEFAULT
    DA_CACHE_ROOT = MYDRIVE_DATA / f"cache/{_DA_CACHE_NAME}"
    GNN_CACHE_ROOT = MYDRIVE_DATA / f"cache/{_GNN_CACHE_NAME}"
    RUNS_PARENT = MYDRIVE_DATA / "runs"
elif os.name == "nt":
    CHUNK_PARENT = next((p for p in WIN_CHUNK_CANDIDATES if p.is_dir()), WIN_CHUNK_CANDIDATES[0])
    DA_CACHE_ROOT = REPO / f"datasets_gnn2_from pc/cache/{_DA_CACHE_NAME}"
    GNN_CACHE_ROOT = REPO / f"datasets_gnn2_from pc/cache/{_GNN_CACHE_NAME}"
    RUNS_PARENT = REPO / "datasets_gnn2_from pc/runs"
else:
    CHUNK_PARENT = REPO / f"datasets_gnn2_from pc/{CHUNK_NAME}"
    DA_CACHE_ROOT = REPO / f"datasets_gnn2_from pc/cache/{_DA_CACHE_NAME}"
    GNN_CACHE_ROOT = REPO / f"datasets_gnn2_from pc/cache/{_GNN_CACHE_NAME}"
    RUNS_PARENT = REPO / "datasets_gnn2_from pc/runs"

_colab_fb = COLAB_CHUNK_DEFAULT if _on_colab() else None
chunk_parent = _resolve_data_path(CHUNK_PARENT, label="CHUNK_PARENT", colab_fallback=_colab_fb)
if SMOKE_TEST:
    CHUNK_GLOB = _smoke_chunk_subdir_glob(chunk_parent, SMOKE_CHUNK_COUNT)
    EPOCHS = SMOKE_EPOCHS
    PATIENCE = SMOKE_PATIENCE
else:
    CHUNK_GLOB = "run_*"
    EPOCHS = 200
    PATIENCE = 30

da_cache_root = _resolve_data_path(
    DA_CACHE_ROOT,
    label="DA_CACHE_ROOT",
    colab_fallback=MYDRIVE_DATA / f"cache/{_DA_CACHE_NAME}" if _on_colab() else None,
)
gnn_cache_root = _resolve_data_path(
    GNN_CACHE_ROOT,
    label="GNN_CACHE_ROOT",
    colab_fallback=MYDRIVE_DATA / f"cache/{_GNN_CACHE_NAME}" if _on_colab() else None,
)
runs_parent = _resolve_data_path(
    RUNS_PARENT,
    label="RUNS_PARENT",
    colab_fallback=MYDRIVE_DATA / "runs" if _on_colab() else None,
)

if not chunk_parent.is_dir():
    raise FileNotFoundError(
        f"CHUNK_PARENT not found:\n  {chunk_parent}\n"
        "Finish the Y-edge stamp cell first (creates *_fixedctrlinit_yedges)."
    )

_ensure_mvagg_under(chunk_parent, REPO)
node_pe = _find_node_pe_csv(chunk_parent)
runs_parent.mkdir(parents=True, exist_ok=True)
da_cache_root.mkdir(parents=True, exist_ok=True)

print("=== Preflight (MT-GPS CCE + fixedctrlinit_yedges) ===")
print(f"REPO:              {REPO}")
print(f"VARIANT:           fixedctrlinit_yedges_cce")
print(f"global_attn_mode:  {GLOBAL_ATTN_MODE}")
print(f"lambda_cap/reg/pv: {LAMBDA_CAP}/{LAMBDA_REG}/{LAMBDA_PV}")
print(f"arch:              h={CCE_HIDDEN} L={CCE_LAYERS} heads={CCE_HEADS} node_emb={CCE_NODE_EMB_DIM}")
print(f"batch_size:        {BATCH_SIZE}")
print(f"SMOKE_TEST:        {SMOKE_TEST}")
print(f"CHUNK_PARENT:      {chunk_parent}")
print(f"DA_CACHE_ROOT:     {da_cache_root}")
print(f"GNN_CACHE_ROOT:    {gnn_cache_root}")
print(f"node_pe_csv:       {node_pe}")

tag = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = runs_parent / (
    f"da_gps_fixedctrlinit_yedges_l{CCE_LAYERS}_h{CCE_HIDDEN}_mvagg_gine_metaaux_regce"
    f"{'_smoke' if SMOKE_TEST else ''}_{tag}"
)
out_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "-u", "train_da_gps_multitask_complex_voltage_gine.py",
    "--chunk_parent", str(chunk_parent),
    "--chunk_subdir_glob", CHUNK_GLOB,
    "--nodes_csv", "gnn_node_features_and_targets_mvagg.csv",
    "--edge_catalog_csv", "gnn_edges_phase_static.csv",
    "--meta_csv", "gnn_sample_meta.csv",
    "--node_feature_cols", "p_load_kw,q_load_kvar,p_pv_kw",
    "--exclude_bess_features",
    "--node_pe_csv", str(node_pe),
    "--node_pe_cols", "auto",
    "--n_system_tokens", "10",
    "--aux_meta_cols", META_AUX_COLS,
    "--lambda_pv", LAMBDA_PV,
    "--out_dir", str(out_dir),
    "--cache_dir", str(da_cache_root),
    "--bootstrap_gnn_cache_dir", str(gnn_cache_root),
    "--epochs", str(EPOCHS),
    "--batch_size", str(BATCH_SIZE),
    "--hidden", str(CCE_HIDDEN),
    "--layers", str(CCE_LAYERS),
    "--heads", str(CCE_HEADS),
    "--node_emb_dim", str(CCE_NODE_EMB_DIM),
    "--edge_emb_dim", "0",
    "--lr", "5e-4",
    "--weight_decay", "1e-5",
    "--lambda_cap", LAMBDA_CAP,
    "--lambda_reg", LAMBDA_REG,
    "--reg_loss", "ce",
    "--per_device_cap_head",
    "--per_device_reg_head",
    "--global_attn_mode", GLOBAL_ATTN_MODE,
    "--patience", str(PATIENCE),
    "--seed", str(SMOKE_SEED),
    "--train_frac", "0.80",
    "--val_frac", "0.10",
    "--sample_frac", "1.0",
    "--num_workers", str(NUM_WORKERS),
    "--log_every", "10",
    "--checkpoint_every", "10",
    "--early_stop_on", "total",
    "--dropout", "0.1",
]

print("\nRunning:\n ", " ".join(cmd), "\n", flush=True)

with subprocess.Popen(
    cmd,
    cwd=str(REPO),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
    text=True,
    encoding="utf-8",
    errors="replace",
) as proc:
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()

if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)

print("\nTraining completed.")
print("Run dir:", out_dir.resolve())
print("Checkpoint (best):", (out_dir / "da_gps_multitask_best.pt").resolve())
print("Checkpoint (last):", (out_dir / "training_last.pt").resolve())
print("Report:", (out_dir / "da_gps_report.json").resolve())


### VREG3-A attention on four representative days (peak-load + median)

Same topology style as the paper mean-attention map, but **one figure per day** under
`a representativ days/` (`load_day_00{1..4}` + `irr_day_00{1..4}`).

For each day this cell writes **two** maps:

| Tag | Timestep |
|-----|----------|
| `peakload` | argmax of the day load multiplier |
| `median` | step whose load multiplier is closest to the day median |

Node features start from a cache reference sample, then **P/Q load and `p_pv_kw` are
rescaled** by that day's load / irradiance multipliers (same idea as Method A profile
scaling; no per-step OpenDSS). Attention = last-layer **node→token** broadcast to
`reg_vreg3_a_tap_pu`. Color scale is **shared** across all eight figures (absolute).

Outputs under `Figures/day_attention/`.


In [ ]:
# ============================================================
# VREG3-A attention: 4 representative days × {peakload, median}
# ============================================================
from __future__ import annotations

import importlib
import json
import os
import sys
from pathlib import Path

import numpy as np
import torch
from torch_geometric.data import Data

# -------------------- knobs --------------------
REG_COL = "reg_vreg3_a_tap_pu"
LAYER = -1  # last GPS layer
ATTN_DIR = "node_to_token"  # broadcast
DAYS = (1, 2, 3, 4)
NPTS = 288
STEP_MIN = 5
SCENARIO_SCALE = 1.0
DAILY_STRESS = 0.0
BUS_COLOR_CMAP = "YlOrRd"
SHOW_PREVIEWS = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Prefer shipped CCE run; fall back to any attention checkpoint with norms + best pt
RUN_CANDIDATES = [
    Path(r"C:\Users\alita\OneDrive\Desktop\GNN2\gnn2_architecture_search\attention checkpoints\da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE"),
    Path("/content/drive/MyDrive/datasets_gnn2/runs/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE"),
    Path("/content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE"),
]
CACHE_CANDIDATES = [
    Path(r"C:\Users\alita\OneDrive\Desktop\GNN2\datasets_gnn2_from pc\run_001_ref0_slim__full__nobess__regce__mauxb7bd1d58.pt"),
    Path(r"C:\Users\alita\OneDrive\Desktop\GNN2\datasets_gnn2_from pc\run_001_scen_0000_0049_seed_20420233__full__nobess__regce__mauxb7bd1d58.pt"),
    Path("/content/GNN-Sandia/datasets_gnn2_from pc/run_001_ref0_slim__full__nobess__regce__mauxb7bd1d58.pt"),
    Path("/content/drive/MyDrive/datasets_gnn2/cache/da_gps_chunked_mvagg_full_gine"),
]
EDGE_CANDIDATES = [
    Path(r"C:\Users\alita\OneDrive\Desktop\GNN2\datasets_gnn2_from pc\original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40\run_001_scen_0000_0049_seed_20420233\gnn_edges_phase_static.csv"),
    Path("/content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_001_scen_0000_0049_seed_20420233/gnn_edges_phase_static.csv"),
    Path("/content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40_fixedctrlinit_yedges/run_001_scen_0000_0049_seed_20420233/gnn_edges_phase_static.csv"),
]
# ------------------------------------------------


def _find_repo() -> Path:
    for p in [
        Path(os.environ.get("GNN2_REPO_ROOT", "")),
        Path.cwd(),
        Path(r"C:\Users\alita\OneDrive\Desktop\GNN2"),
        Path("/content/GNN-Sandia"),
        Path("/content/GNN2"),
    ]:
        if not p or not str(p):
            continue
        p = Path(p).resolve()
        if (p / "extract_da_gps_attention.py").is_file() and (
            p / "plot_ieee8500_feeder_topology.py"
        ).is_file():
            return p
    raise FileNotFoundError("Repo root with extract_da_gps_attention.py not found")


REPO = _find_repo()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import plot_ieee8500_feeder_topology as p8500
from compare_opendss_snapshot_helpers import prepare_parity_profiles
from extract_da_gps_attention import (
    _assert_x_zscore_shapes,
    _build_model,
    _infer_node_in_edge_dim,
    _load_eval_edges,
    _node_order_from_ntl,
    augment_da_gps_pack_for_eval,
)

p8500 = importlib.reload(p8500)
p8500.clear_icon_caches()


def _first_file(cands: list[Path]) -> Path:
    for p in cands:
        if p.is_file():
            return p.resolve()
    raise FileNotFoundError("none of:\n  " + "\n  ".join(str(c) for c in cands))


def _first_run_dir(cands: list[Path]) -> Path:
    for p in cands:
        if (p / "da_gps_multitask_best.pt").is_file() or (p / "training_last.pt").is_file():
            return p.resolve()
        if (p / "x_mean.pt").is_file():
            return p.resolve()
    raise FileNotFoundError("no usable RUN_DIR among candidates")


def _resolve_cache(cands: list[Path]) -> Path:
    for p in cands:
        if p.is_file() and p.suffix == ".pt":
            return p.resolve()
        if p.is_dir():
            hits = sorted(p.glob("run_001*.pt")) + sorted(p.glob("*.pt"))
            if hits:
                return hits[0].resolve()
    raise FileNotFoundError("no cache .pt found")


def _ckpt_in(run: Path) -> Path:
    for name in ("da_gps_multitask_best.pt", "training_last.pt"):
        p = run / name
        if p.is_file():
            return p
    raise FileNotFoundError(f"no checkpoint under {run}")


def _phase_node_to_bus(node: str) -> str:
    s = str(node).strip()
    if "." in s:
        bus, ph = s.rsplit(".", 1)
        if ph.isdigit():
            return bus
    return s


def _phase_suffix(node: str) -> int | None:
    s = str(node).strip()
    if "." in s:
        _, ph = s.rsplit(".", 1)
        if ph.isdigit():
            return int(ph)
    return None


def _bus_vals_phase_a(a_nodes: np.ndarray, node_names: list[str]) -> dict[str, float]:
    out: dict[str, float] = {}
    for nm, a in zip(node_names, a_nodes):
        if _phase_suffix(nm) != 1:
            continue
        out[_phase_node_to_bus(nm)] = float(a)
    if not out:
        raise RuntimeError("No phase-A buses in attention vector")
    return out


def _day_step_indices(m_load: np.ndarray) -> dict[str, int]:
    m = np.asarray(m_load, dtype=float).reshape(-1)
    peak_i = int(np.nanargmax(m))
    med = float(np.nanmedian(m))
    med_i = int(np.nanargmin(np.abs(m - med)))
    return {"peakload": peak_i, "median": med_i}


RUN_DIR = _first_run_dir(RUN_CANDIDATES)
CKPT = _ckpt_in(RUN_DIR)
CACHE_PT = _resolve_cache(CACHE_CANDIDATES)
EDGES_CSV = _first_file(EDGE_CANDIDATES)
DAY_DIR = REPO / "a representativ days"
FIGURES_DIR = REPO / "Figures" / "day_attention"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
ICONS = REPO / "outputs" / "Icons"

print("REPO:", REPO)
print("RUN_DIR:", RUN_DIR)
print("CKPT:", CKPT.name)
print("CACHE:", CACHE_PT)
print("EDGES:", EDGES_CSV)
print("DAY_DIR:", DAY_DIR)
print("OUT:", FIGURES_DIR)
print("DEVICE:", DEVICE)

pack = torch.load(CKPT, map_location="cpu", weights_only=False)
sd = pack["model_state_dict"]
z = torch.load(CACHE_PT, map_location="cpu", weights_only=False)
node_to_local = z["node_to_local"]
x_cache = z["x"].float()
node_order = _node_order_from_ntl(node_to_local)
ref_i = 0
ref_x = x_cache[ref_i].numpy().astype(np.float32, copy=True)

x_mean = torch.load(RUN_DIR / "x_mean.pt", map_location="cpu", weights_only=True).float()
x_std = torch.load(RUN_DIR / "x_std.pt", map_location="cpu", weights_only=True).float()
_assert_x_zscore_shapes(x_cache, x_mean, x_std, run_dir=RUN_DIR, cache_pt=CACHE_PT)

augment_da_gps_pack_for_eval(pack, sd, CKPT, RUN_DIR, node_in_dim_hint=int(x_cache.shape[-1]))
hidden = int(pack["hidden"])
node_emb_dim = int(pack["node_emb_dim"])
edge_emb_dim = int(pack["edge_emb_dim"])
_, raw_edge_dim = _infer_node_in_edge_dim(
    sd, hidden=hidden, node_emb_dim=node_emb_dim, edge_emb_dim=edge_emb_dim
)
edge_index, edge_attr = _load_eval_edges(
    EDGES_CSV, node_to_local, raw_edge_dim=int(raw_edge_dim), cache_z=z, sd=sd
)
model = _build_model(pack, sd, num_edges=int(edge_index.shape[1]), dropout=0.0)
model.load_state_dict(sd, strict=True)
model.to(DEVICE)
model.eval()

feat_cols = list(pack.get("node_feature_cols") or ["p_load_kw", "q_load_kvar", "p_pv_kw"])
# PE dims sit after raw features in x; scale only the first 3 load/PV channels of the feature block
# Training pack often stores pe separately already merged into x — use report/hp if present
hp = pack.get("hparams") or pack.get("args") or {}
if isinstance(hp, dict) and hp.get("node_feature_cols"):
    feat_cols = [str(c) for c in str(hp["node_feature_cols"]).split(",") if str(c).strip()]
# Fallback: first 3 columns are p_load, q_load, p_pv in CCE
col_p, col_q, col_pv = 0, 1, 2
print("feature scale cols (assumed):", col_p, col_q, col_pv, "feat_cols=", feat_cols[:6])

reg_cols = list(pack.get("reg_target_cols") or [])
cap_cols = list(pack.get("cap_target_cols") or [])
if not reg_cols:
    # recover from report
    rep = RUN_DIR / "da_gps_report.json"
    if rep.is_file():
        rj = json.loads(rep.read_text(encoding="utf-8"))
        reg_cols = list(rj.get("reg_target_cols") or rj.get("hparams", {}).get("reg_cols") or [])
        cap_cols = list(rj.get("cap_target_cols") or [])
if REG_COL not in reg_cols:
    # common CCE order
    reg_cols = reg_cols or [
        "reg_feeder_rega_tap_pu",
        "reg_feeder_regb_tap_pu",
        "reg_feeder_regc_tap_pu",
        "reg_vreg2_a_tap_pu",
        "reg_vreg2_b_tap_pu",
        "reg_vreg2_c_tap_pu",
        "reg_vreg3_a_tap_pu",
        "reg_vreg3_b_tap_pu",
        "reg_vreg3_c_tap_pu",
        "reg_vreg4_a_tap_pu",
        "reg_vreg4_b_tap_pu",
        "reg_vreg4_c_tap_pu",
    ]
    cap_cols = cap_cols or [
        "cap_capbank0a_n_steps_on",
        "cap_capbank0b_n_steps_on",
        "cap_capbank0c_n_steps_on",
        "cap_capbank1a_n_steps_on",
        "cap_capbank1b_n_steps_on",
        "cap_capbank1c_n_steps_on",
        "cap_capbank2a_n_steps_on",
        "cap_capbank2b_n_steps_on",
        "cap_capbank2c_n_steps_on",
        "cap_capbank3_n_steps_on",
    ]
if REG_COL not in reg_cols:
    raise ValueError(f"{REG_COL} not in reg_cols={reg_cols}")
tok_idx = len(cap_cols) + int(reg_cols.index(REG_COL))
print(f"token {REG_COL} -> idx {tok_idx} (n_cap={len(cap_cols)})")

dev = torch.device(DEVICE)
x_mean_d = x_mean.to(dev)
x_std_d = x_std.to(dev)
edge_index_d = edge_index.to(dev)
edge_attr_d = edge_attr.to(dev)


@torch.no_grad()
def _attn_nodes_for_x(x_np: np.ndarray) -> np.ndarray:
    x_t = torch.from_numpy(np.asarray(x_np, dtype=np.float32))
    x_n = ((x_t - x_mean_d.cpu()) / x_std_d.cpu()).to(device=dev, dtype=torch.float32)
    data = Data(x=x_n, edge_index=edge_index_d, edge_attr=edge_attr_d)
    data.num_graphs = 1
    out = model.forward_node_to_token_attention(data)
    if len(out) >= 5:
        layer_probs_nt = out[0]
    else:
        raise RuntimeError(f"unexpected attention return len={len(out)}")
    # list of (H,N,T) or (1,H,N,T)
    layer = layer_probs_nt[int(LAYER) if int(LAYER) >= 0 else -1]
    a = layer.detach().float().cpu().numpy()
    while a.ndim > 2:
        # mean over heads / batch
        a = a.mean(axis=0)
    # a: (N, T)
    return np.asarray(a[:, int(tok_idx)], dtype=np.float64)


def _scale_ref_x(m_t: float, ir_t: float, *, m_ref: float, ir_ref: float) -> np.ndarray:
    x = ref_x.copy()
    ms = float(m_t) / max(float(m_ref), 1e-6)
    irs = float(ir_t) / max(float(ir_ref), 1e-6)
    x[:, col_p] *= ms
    x[:, col_q] *= ms
    x[:, col_pv] *= irs
    return x


# Collect bus maps first (for shared color scale), then plot
jobs: list[dict] = []
for day in DAYS:
    load_csv = DAY_DIR / f"load_day_{day:03d}.csv"
    irr_csv = DAY_DIR / f"irr_day_{day:03d}.csv"
    if not load_csv.is_file() or not irr_csv.is_file():
        raise FileNotFoundError(f"Missing day profiles:\n  {load_csv}\n  {irr_csv}")
    prof = prepare_parity_profiles(
        load_csv=load_csv,
        irr_csv=irr_csv,
        npts=int(NPTS),
        step_min=float(STEP_MIN),
        daily_stress=float(DAILY_STRESS),
    )
    m_eff = np.asarray(prof.m_eff, dtype=float)
    m_irr = np.asarray(prof.m_irr, dtype=float)
    steps = _day_step_indices(m_eff)
    m_ref = float(np.nanmean(m_eff))
    ir_ref = float(np.nanmean(m_irr))
    print(
        f"day {day}: peakload step={steps['peakload']} "
        f"(m={m_eff[steps['peakload']]:.4g})  "
        f"median step={steps['median']} (m={m_eff[steps['median']]:.4g})"
    )
    for tag, si in steps.items():
        m_t = float(m_eff[si]) * float(SCENARIO_SCALE)
        ir_t = float(m_irr[si])
        x_step = _scale_ref_x(m_t, ir_t, m_ref=m_ref, ir_ref=ir_ref)
        a_nodes = _attn_nodes_for_x(x_step)
        bus_vals = _bus_vals_phase_a(a_nodes, node_order)
        jobs.append(
            {
                "day": day,
                "tag": tag,
                "step": si,
                "m_t": m_t,
                "ir_t": ir_t,
                "bus_vals": bus_vals,
                "a_nodes": a_nodes,
            }
        )

all_vals = np.concatenate([np.asarray(list(j["bus_vals"].values()), dtype=float) for j in jobs])
vmin = float(np.min(all_vals))
vmax = float(np.max(all_vals))
print(f"shared color scale: [{vmin:.6g}, {vmax:.6g}] over {len(jobs)} maps")

written = []
for j in jobs:
    day, tag = int(j["day"]), str(j["tag"])
    out_base = f"vreg3_a_attention_day{day:02d}_{tag}"
    label = (
        f"Broadcast attn → VREG3A  |  day {day} {tag}  "
        f"(t={j['step']}, m={j['m_t']:.3g}, irr={j['ir_t']:.3g})"
    )
    paths = p8500.plot_ieee8500_feeder_topology(
        dss_dir=REPO / "8500 nodes with solar unbalanced",
        out_dir=FIGURES_DIR,
        out_basename=out_base,
        style="paper",
        line_width_mode="ampacity",
        taper_line_width=False,
        use_opendss_power=False,
        ampacity_thickness_min=0.50,
        ampacity_thickness_max=7.0,
        ampacity_gamma=1.0,
        ampacity_smooth_blend=0.35,
        ampacity_downstream_weight=0.55,
        show_device_icons=True,
        paper_device_icons_only=True,
        show_loads=False,
        show_substation_icon=False,
        show_capacitor_icons=False,
        show_regulator_icons=True,
        regulator_name_filter=["vreg3_a"],
        bus_color_values=j["bus_vals"],
        bus_color_cmap=BUS_COLOR_CMAP,
        bus_color_point_size=7.0,
        bus_color_alpha=0.90,
        bus_color_log=False,
        bus_color_colorbar=True,
        bus_color_label=label,
        bus_color_vmin=vmin,
        bus_color_vmax=vmax,
        bus_color_upper_frac=None,
        bus_color_power_gamma=1.0,
        device_icon_paths={
            "substation": ICONS / "substation.svg",
            "regulator": ICONS / "voltage regulator.svg",
            "capacitor": ICONS / "capacitor bank.svg",
        },
        icon_size_scale=2.6,
    )
    written.append(paths)
    print("wrote", paths.get("pdf") or paths)
    if SHOW_PREVIEWS:
        try:
            from IPython.display import Image, display
            import fitz

            pdf = Path(paths["pdf"])
            doc = fitz.open(pdf)
            pix = doc[0].get_pixmap(matrix=fitz.Matrix(1.5, 1.5), alpha=False)
            preview = pdf.with_name(pdf.stem + "_preview.png")
            pix.save(preview)
            display(Image(filename=str(preview)))
        except Exception as exc:  # noqa: BLE001
            print("preview skipped:", exc)

# also dump npz of node attentions for reuse
npz_path = FIGURES_DIR / "vreg3_a_day_peakload_median_attn.npz"
np.savez_compressed(
    npz_path,
    node_names=np.asarray(node_order, dtype=object),
    days=np.asarray([j["day"] for j in jobs], dtype=np.int32),
    tags=np.asarray([j["tag"] for j in jobs], dtype=object),
    steps=np.asarray([j["step"] for j in jobs], dtype=np.int32),
    attn=np.stack([j["a_nodes"] for j in jobs], axis=0),
    vmin=vmin,
    vmax=vmax,
)
print("saved", npz_path)
print(f"Done: {len(written)} figures under {FIGURES_DIR}")
